In [1]:
import sys
import os

# Add the parent directory one level up to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

# Import the initialize_spark function from the package
from notebooks.config.initialize_spark import *

# Initialize the Spark session and delta table path
spark, delta_lake_path = initialize_spark(use_delta=True)

initialize_spark() executed from: /home/jovyan/work/notebooks/config/initialize_spark.py
DATA_FOLDER being defined as: /home/jovyan/work/data
:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9f190c1d-1964-4c06-9c71-a49f524638d3;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-core_2.12/2.4.0/delta-core_2.12-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-core_2.12;2.4.0!delta-core_2.12.jar (971ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/2.4.0/delta-storage-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;2.4.0!delta-storage.jar (50ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (99ms)
:: resolution report :: resolve 1868ms :: artifac

Delta Lake support is enabled.


In [2]:
!ls -l {delta_lake_path}

total 4
drwxrwxrwx 3 jovyan users 4096 Aug 24 16:56 netflix_titles


In [4]:
# Construct the SQL command as a string using the variable
#spark.sql(f"DROP TABLE IF EXISTS default.netflix_titles")

create_table_query = f"""
CREATE TABLE default.netflix_titles (
    show_id STRING,
    type STRING,
    title STRING,
    director STRING,
    cast STRING,
    country STRING,
    date_added STRING,
    release_year STRING,
    rating STRING,
    duration STRING,
    listed_in STRING,
    description STRING 
) USING DELTA LOCATION '{delta_lake_path}/netflix_titles';
"""

# Execute the SQL command
spark.sql(create_table_query)


24/08/24 17:24:58 INFO SharedState: Setting hive.metastore.warehouse.dir ('null') to the value of spark.sql.warehouse.dir.
24/08/24 17:24:58 INFO SharedState: Warehouse path is 'file:/home/jovyan/work/notebooks/Chapter03/spark-warehouse'.
24/08/24 17:24:59 INFO StandaloneSchedulerBackend$StandaloneDriverEndpoint: Registered executor NettyRpcEndpointRef(spark-client://Executor) (172.18.0.4:52406) with ID 2,  ResourceProfileId 0
24/08/24 17:24:59 INFO StandaloneSchedulerBackend$StandaloneDriverEndpoint: Registered executor NettyRpcEndpointRef(spark-client://Executor) (172.18.0.3:57470) with ID 0,  ResourceProfileId 0
24/08/24 17:24:59 INFO BlockManagerMasterEndpoint: Registering block manager 172.18.0.3:34973 with 434.4 MiB RAM, BlockManagerId(0, 172.18.0.3, 34973, None)
24/08/24 17:24:59 INFO BlockManagerMasterEndpoint: Registering block manager 172.18.0.4:38117 with 434.4 MiB RAM, BlockManagerId(2, 172.18.0.4, 38117, None)
24/08/24 17:24:59 INFO StandaloneSchedulerBackend$StandaloneDri

DataFrame[]

In [5]:
df = (spark.read
      .format("csv")
      .option("header", "true")
      .load(f"{DATA_FOLDER}/netflix_titles.csv"))


24/08/24 17:25:08 INFO InMemoryFileIndex: It took 47 ms to list leaf files for 1 paths.
24/08/24 17:25:08 INFO BlockManagerInfo: Removed broadcast_1_piece0 on 7c457db024e6:42991 in memory (size: 15.5 KiB, free: 434.4 MiB)
24/08/24 17:25:08 INFO BlockManagerInfo: Removed broadcast_1_piece0 on 172.18.0.4:38117 in memory (size: 15.5 KiB, free: 434.4 MiB)
24/08/24 17:25:08 INFO BlockManagerInfo: Removed broadcast_0_piece0 on 7c457db024e6:42991 in memory (size: 35.2 KiB, free: 434.4 MiB)
24/08/24 17:25:08 INFO BlockManagerInfo: Removed broadcast_0_piece0 on 172.18.0.4:38117 in memory (size: 35.2 KiB, free: 434.4 MiB)
24/08/24 17:25:08 INFO InMemoryFileIndex: It took 1 ms to list leaf files for 1 paths.
24/08/24 17:25:08 INFO FileSourceStrategy: Pushed Filters: 
24/08/24 17:25:08 INFO FileSourceStrategy: Post-Scan Filters: (length(trim(value#37, None)) > 0)
24/08/24 17:25:09 INFO CodeGenerator: Code generated in 15.980762 ms
24/08/24 17:25:09 INFO MemoryStore: Block broadcast_2 stored as val

In [6]:
df.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [7]:
spark.sql("DROP TABLE IF EXISTS default.netflix_titles")

DataFrame[]

In [9]:


# Write the DataFrame to a Delta table
df.write.format("delta") \
    .mode("overwrite") \
    .option("path", f"/home/jovyan/delta_lake/netflix_titles") \
    .saveAsTable("default.netflix_titles")


24/08/24 17:26:53 INFO FileSourceStrategy: Pushed Filters: 
24/08/24 17:26:53 INFO FileSourceStrategy: Post-Scan Filters: 
24/08/24 17:26:53 INFO FileSourceStrategy: Pushed Filters: 
24/08/24 17:26:53 INFO FileSourceStrategy: Post-Scan Filters: 
24/08/24 17:26:53 INFO ParquetUtils: Using default output committer for Parquet: org.apache.parquet.hadoop.ParquetOutputCommitter
24/08/24 17:26:53 INFO MemoryStore: Block broadcast_7 stored as values in memory (estimated size 202.2 KiB, free 433.0 MiB)
24/08/24 17:26:53 INFO MemoryStore: Block broadcast_7_piece0 stored as bytes in memory (estimated size 35.1 KiB, free 433.0 MiB)
24/08/24 17:26:53 INFO BlockManagerInfo: Added broadcast_7_piece0 in memory on 7c457db024e6:42991 (size: 35.1 KiB, free: 434.1 MiB)
24/08/24 17:26:53 INFO SparkContext: Created broadcast 7 from $anonfun$recordDeltaOperationInternal$1 at DatabricksLogging.scala:128
24/08/24 17:26:53 INFO FileSourceScanExec: Planning scan with bin packing, max size: 4194304 bytes, open c

In [ ]:
# Load the Spark SQL Magic extension
%load_ext sparksql_magic

In [12]:
%%sparksql 
SELECT * FROM default.netflix_titles LIMIT 3;

24/08/24 17:30:55 INFO PrepareDeltaScan: DELTA: Filtering files for query
24/08/24 17:30:55 INFO CodeGenerator: Code generated in 138.355394 ms
24/08/24 17:30:55 INFO CodeGenerator: Code generated in 29.123151 ms
24/08/24 17:30:56 INFO SparkContext: Starting job: take at /opt/conda/lib/python3.11/site-packages/sparksql_magic/sparksql.py:84
24/08/24 17:30:56 INFO DAGScheduler: Got job 12 (take at /opt/conda/lib/python3.11/site-packages/sparksql_magic/sparksql.py:84) with 50 output partitions
24/08/24 17:30:56 INFO DAGScheduler: Final stage: ResultStage 20 (take at /opt/conda/lib/python3.11/site-packages/sparksql_magic/sparksql.py:84)
24/08/24 17:30:56 INFO DAGScheduler: Parents of final stage: List(ShuffleMapStage 19)
24/08/24 17:30:56 INFO DAGScheduler: Missing parents: List()
24/08/24 17:30:56 INFO DAGScheduler: Submitting ResultStage 20 (MapPartitionsRDD[63] at take at /opt/conda/lib/python3.11/site-packages/sparksql_magic/sparksql.py:84), which has no missing parents
24/08/24 17:30:

show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,null,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable."
s2,TV Show,Blood & Water,null,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno Greeff, Xolile Tshabalala, Getmore Sithole, Cindy Mahlangu, Ryle De Morny, Greteli Fincham, Sello Maake Ka-Ncube, Odwa Gwanya, Mekaila Mathys, Sandi Schultz, Duane Williams, Shamilla Miller, Patrick Mofokeng",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth."
s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari, Sofia Lesaffre, Salim Kechiouche, Noureddine Farihi, Geert Van Rampelberg, Bakary Diombera",null,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Action & Adventure","To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robbers are pulled into a violent and deadly turf war."


In [13]:
spark.stop()

24/08/24 17:31:11 INFO SparkContext: SparkContext is stopping with exitCode 0.
24/08/24 17:31:11 INFO SparkUI: Stopped Spark web UI at http://7c457db024e6:4040
24/08/24 17:31:11 INFO StandaloneSchedulerBackend: Shutting down all executors
24/08/24 17:31:11 INFO StandaloneSchedulerBackend$StandaloneDriverEndpoint: Asking each executor to shut down
24/08/24 17:31:11 INFO MapOutputTrackerMasterEndpoint: MapOutputTrackerMasterEndpoint stopped!
24/08/24 17:31:11 INFO MemoryStore: MemoryStore cleared
24/08/24 17:31:11 INFO BlockManager: BlockManager stopped
24/08/24 17:31:11 INFO BlockManagerMaster: BlockManagerMaster stopped
24/08/24 17:31:11 INFO OutputCommitCoordinator$OutputCommitCoordinatorEndpoint: OutputCommitCoordinator stopped!
24/08/24 17:31:11 INFO SparkContext: Successfully stopped SparkContext
24/08/24 20:07:30 INFO ShutdownHookManager: Shutdown hook called
24/08/24 20:07:30 INFO ShutdownHookManager: Deleting directory /tmp/spark-4351f440-e77e-4d27-aff5-177475e8e9b8/pyspark-e16c